# GradTree experiments: all datasets, GradTree-only plots

Train the PyTorch GradTree on **Adult, Echocardiogram, Iris, Spambase, and Wine**, then produce GradTree-only versions of the reference figures:

1. **Feature-selection ablation** — ST entmax vs ST softmax vs sparsemax.
2. **Main results** — test macro F1, tree size after pruning, and overfitting.
3. **Depth sweep** — test macro F1 and tree size vs maximum depth.

CART and Random Forest are intentionally excluded.

## Before running

- Run the notebook from the repository root, or set `PROJECT_ROOT` manually after the imports cell.
- Install dependencies from the repository root:
  ```bash
  pip install -r model/sunny/requirements.txt
  ```
  On Colab, torch is usually preinstalled, so `pip install -q entmax` is often enough.
- All five datasets need `preprocess/<dataset>_output/seed_42` through `seed_51`. The dataset-check cell reports anything missing.
- A GPU is strongly recommended for the paper protocol.

## Quick vs paper protocol

`QUICK_RUN = True` uses 2 seeds and short training so you can validate the notebook end-to-end. Set `QUICK_RUN = False` for the paper protocol:

```text
10 seeds (42–51), 10,000 max epochs, patience 200, 3 restarts
```

Every run is cached in `model/sunny/results/runs.csv`, scoped to the current protocol, so interrupted sweeps resume from completed rows without mixing quick and paper results.

## Scope notes

- The model supports `entmax`, `softmax`, and `sparsemax`. The paper's Gumbel-softmax and temperature-annealing variants are not implemented here.
- Paper references come from Tables 1/2 (test macro F1), 9 (tree size), 12 (settings and tuned depth), and 5 (ablation).
- Each trial uses the test split from its data seed (42 + trial), with model seed 42. Per-dataset results are mean ± std over trials; the paper does not average macro F1 across datasets.


In [ ]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import NullFormatter, ScalarFormatter
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "model" / "sunny").is_dir() and (candidate / "preprocess").is_dir():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model.sunny.config import GradTreeConfig
from model.sunny.data import load_dataset
from model.sunny.gradtree import GradTree
from model.sunny.metrics import macro_f1_score
from model.sunny.trainer import GradTreeTrainer

print("Project root:", PROJECT_ROOT)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
import hashlib

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Quick mode validates the notebook; set False for the paper protocol.
QUICK_RUN = True
RUN_MAIN = True
RUN_ABLATION = True
RUN_DEPTH_SWEEP = True   # expensive; set False to skip the third figure

if QUICK_RUN:
    TRIAL_SEEDS = [42, 43]
    EPOCHS = 200
    PATIENCE = 20
    RESTARTS = 0
else:
    TRIAL_SEEDS = list(range(42, 52))
    EPOCHS = 10_000
    PATIENCE = 200
    RESTARTS = 3

DATASETS = ["adult", "echocardiogram", "iris", "spambase", "wine"]
DATASET_LABELS = {
    "adult": "Adult",
    "echocardiogram": "Echocardiogram",
    "iris": "Iris",
    "spambase": "Spambase",
    "wine": "Wine",
}

# Appendix C, Table 12: optimized per-dataset settings.
PAPER_SETTINGS = {
    "adult": dict(depth=8, lr_index=0.050, lr_values=0.005, lr_leaf=0.050, loss="poly", poly_epsilon=5),
    "echocardiogram": dict(depth=8, lr_index=0.010, lr_values=0.050, lr_leaf=0.100, loss="poly", poly_epsilon=5),
    "iris": dict(depth=7, lr_index=0.005, lr_values=0.005, lr_leaf=0.05, loss="crossentropy", poly_epsilon=2),
    "spambase": dict(depth=10, lr_index=0.005, lr_values=0.010, lr_leaf=0.010, loss="crossentropy", poly_epsilon=2),
    "wine": dict(depth=10, lr_index=0.010, lr_values=0.050, lr_leaf=0.01, loss="focal", poly_epsilon=2, focal_gamma=3.0),
}

# Paper reference values: (mean, std).
PAPER_TEST_F1 = {
    "adult": (0.743, 0.034),
    "echocardiogram": (0.658, 0.113),
    "iris": (0.938, 0.057),
    "spambase": (0.903, 0.025),
    "wine": (0.933, 0.031),
}
PAPER_TREE_SIZE = {
    "adult": (128.00, 37.19),
    "echocardiogram": (43.20, 12.82),
    "iris": (14.40, 2.54),
    "spambase": (120.20, 37.01),
    "wine": (24.00, 4.12),
}

ABLATION_DATASETS = ["echocardiogram", "iris", "wine"]
ABLATION_ACTIVATIONS = ["entmax", "softmax", "sparsemax"]
ABLATION_LABELS = {
    "entmax": "ST entmax (GradTree)",
    "softmax": "ST softmax",
    "sparsemax": "ST sparsemax",
}
DEPTHS = [1, 2, 4, 6, 8, 10]

# Cache key: quick/paper, training length and the per-dataset settings.
SETTINGS_HASH = hashlib.md5(
    json.dumps(PAPER_SETTINGS, sort_keys=True).encode()
).hexdigest()[:8]
PROTOCOL = (
    f"{'quick' if QUICK_RUN else 'paper'}"
    f"_e{EPOCHS}_p{PATIENCE}_r{RESTARTS}_{SETTINGS_HASH}"
)

RESULTS_DIR = PROJECT_ROOT / "model" / "sunny" / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RUN_CSV = RESULTS_DIR / "runs.csv"

# Validated categorical slots 1-3 from the visualization palette.
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_2 = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
AXIS = "#c3c2b7"
BLUE = "#2a78d6"
ORANGE = "#eb6834"
AQUA = "#1baf7a"
PAPER = "#52514e"
ACT_COLORS = {"entmax": BLUE, "softmax": ORANGE, "sparsemax": AQUA}

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.edgecolor": AXIS,
    "axes.labelcolor": INK_2,
    "axes.titlecolor": INK,
    "xtick.color": MUTED,
    "ytick.color": INK_2,
    "xtick.major.size": 0,
    "ytick.major.size": 0,
    "text.color": INK,
})

print("Quick run:", QUICK_RUN, "| device:", DEVICE, "| seeds:", TRIAL_SEEDS)
print("Epochs:", EPOCHS, "| patience:", PATIENCE, "| restarts:", RESTARTS)
print("Protocol:", PROTOCOL)
print("Results:", RESULTS_DIR)


In [ ]:
# Dataset check: all five datasets need seed_42 ... seed_51.
missing = []
for dataset in DATASETS:
    for seed in range(42, 52):
        try:
            load_dataset(dataset, seed=seed)
        except Exception as exc:
            missing.append((dataset, seed, str(exc)))

if missing:
    for item in missing:
        print("MISSING:", item)
    raise FileNotFoundError(
        "Some dataset seeds are missing. Re-run the preprocessing notebook "
        "with SAVE_ALL_SEEDS = True before training."
    )
print("All five datasets have seed_42 ... seed_51.")

RESULT_COLUMNS = [
    "protocol", "experiment", "dataset", "seed", "depth", "activation", "loss",
    "test_macro_f1", "test_accuracy", "valid_macro_f1", "train_macro_f1",
    "overfitting", "tree_nodes", "tree_internal", "tree_leaves",
    "tree_test_macro_f1", "best_epoch", "best_restart",
    "best_monitor_loss", "config_json",
]


def read_all_results():
    if RUN_CSV.exists() and RUN_CSV.stat().st_size > 0:
        return pd.read_csv(RUN_CSV)
    return pd.DataFrame(columns=RESULT_COLUMNS)


def read_results():
    df = read_all_results()
    if "protocol" in df.columns:
        df = df[df["protocol"] == PROTOCOL].reset_index(drop=True)
    return df


def append_result(row):
    df = read_all_results()
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(RUN_CSV, index=False)
    return df


def cached_row(experiment, dataset, seed, depth, activation):
    df = read_results()
    if df.empty or not set(RESULT_COLUMNS).issubset(df.columns):
        return None
    mask = (
        (df["experiment"] == experiment)
        & (df["dataset"] == dataset)
        & (df["seed"] == seed)
        & (df["depth"] == depth)
        & (df["activation"] == activation)
    )
    matches = df[mask]
    return None if matches.empty else matches.iloc[-1].to_dict()


def run_one(dataset, seed, depth=None, activation="entmax", experiment="main"):
    settings = PAPER_SETTINGS[dataset]
    run_depth = settings["depth"] if depth is None else int(depth)

    cached = cached_row(experiment, dataset, seed, run_depth, activation)
    if cached is not None:
        print(f"[cached] {experiment} {dataset} seed={seed} depth={run_depth} act={activation}")
        return cached

    splits = load_dataset(dataset, seed=seed)
    config = GradTreeConfig(
        depth=run_depth,
        lr_index=settings["lr_index"],
        lr_values=settings["lr_values"],
        lr_leaf=settings["lr_leaf"],
        batch_size=128,
        epochs=EPOCHS,
        patience=PATIENCE,
        restarts=RESTARTS,
        activation=activation,
        beta=1.0,
        dropout=0.0,
        init="glorot",
        loss=settings["loss"],
        focal_gamma=settings.get("focal_gamma", 3.0),
        focal_alpha=0.5,
        poly_epsilon=settings.get("poly_epsilon", 2.0),
        output_activation="softmax",
        prune=True,
        prune_threshold=1,
        device=DEVICE,
        seed=42,
        verbose=False,
    )
    model = GradTree(
        num_features=splits.num_features,
        num_classes=splits.num_classes,
        depth=config.depth,
        activation=config.activation,
        beta=config.beta,
        dropout=config.dropout,
        init=config.init,
        output_activation=config.output_activation,
        seed=config.seed,
    )
    trainer = GradTreeTrainer(model, config)
    train_result = trainer.fit(
        splits.X_train, splits.y_train, splits.X_valid, splits.y_valid
    )
    train_metrics = trainer.evaluate(splits.X_train, splits.y_train)
    valid_metrics = trainer.evaluate(splits.X_valid, splits.y_valid)
    test_metrics = trainer.evaluate(splits.X_test, splits.y_test)

    tree = model.hard_tree(prune_X=splits.X_train, min_samples=config.prune_threshold)
    tree_internal, tree_leaves = tree.count_nodes()
    tree_test_predictions = tree.predict(splits.X_test)
    tree_test_f1 = macro_f1_score(
        splits.y_test, tree_test_predictions, num_classes=splits.num_classes
    )

    row = {
        "protocol": PROTOCOL,
        "experiment": experiment,
        "dataset": dataset,
        "seed": seed,
        "depth": run_depth,
        "activation": activation,
        "loss": settings["loss"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_accuracy": test_metrics["accuracy"],
        "valid_macro_f1": valid_metrics["macro_f1"],
        "train_macro_f1": train_metrics["macro_f1"],
        "overfitting": train_metrics["macro_f1"] - test_metrics["macro_f1"],
        "tree_nodes": tree_internal + tree_leaves,
        "tree_internal": tree_internal,
        "tree_leaves": tree_leaves,
        "tree_test_macro_f1": tree_test_f1,
        "best_epoch": train_result.best_epoch,
        "best_restart": train_result.best_restart,
        "best_monitor_loss": train_result.best_monitor_loss,
        "config_json": json.dumps(config.to_dict()),
    }
    append_result(row)
    print(
        f"[done] {experiment} {dataset} seed={seed} depth={run_depth} "
        f"act={activation} test_f1={row['test_macro_f1']:.4f} nodes={row['tree_nodes']}"
    )
    return row


def run_grid(tasks):
    errors = []
    for index, task in enumerate(tasks, 1):
        print(f"\n[{index}/{len(tasks)}] {task}")
        try:
            run_one(**task)
        except Exception as exc:
            errors.append((task, repr(exc)))
            print("ERROR:", repr(exc))
    if errors:
        print("\nFailures:")
        for task, error in errors:
            print(task, "->", error)
    return errors


def summarize(df, keys):
    if df.empty:
        return df
    out = (
        df.groupby(keys, dropna=False)
        .agg(
            n=("test_macro_f1", "size"),
            test_macro_f1_mean=("test_macro_f1", "mean"),
            test_macro_f1_std=("test_macro_f1", "std"),
            tree_nodes_mean=("tree_nodes", "mean"),
            tree_nodes_std=("tree_nodes", "std"),
            overfitting_mean=("overfitting", "mean"),
            overfitting_std=("overfitting", "std"),
            tree_test_macro_f1_mean=("tree_test_macro_f1", "mean"),
            tree_test_macro_f1_std=("tree_test_macro_f1", "std"),
        )
        .reset_index()
    )
    for column in (
        "test_macro_f1_std", "tree_nodes_std", "overfitting_std", "tree_test_macro_f1_std"
    ):
        out[column] = out[column].fillna(0.0)
    return out


def style_axes(ax, grid_axis="x"):
    ax.set_facecolor(SURFACE)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(AXIS)
        ax.spines[spine].set_linewidth(1)
    ax.grid(axis=grid_axis, color=GRID, linewidth=1, linestyle="-")
    ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED, labelcolor=INK_2, width=0)


print("Dataset check and helper functions are ready.")
print("Cache filter:", PROTOCOL)


In [ ]:
if QUICK_RUN:
    print("Quick mode: results are for notebook validation, not the final paper numbers.")
else:
    print("Paper protocol: 10 seeds, 10,000 max epochs, patience 200, 3 restarts.")

if RUN_MAIN:
    tasks = [
        dict(experiment="main", dataset=dataset, seed=seed, depth=None, activation="entmax")
        for dataset in DATASETS
        for seed in TRIAL_SEEDS
    ]
    print(f"\n=== Main runs: {len(tasks)} ===")
    run_grid(tasks)
else:
    print("RUN_MAIN = False; skipping main runs.")

if RUN_ABLATION:
    tasks = [
        dict(experiment="ablation", dataset=dataset, seed=seed, depth=None, activation=activation)
        for dataset in ABLATION_DATASETS
        for activation in ABLATION_ACTIVATIONS
        for seed in TRIAL_SEEDS
    ]
    print(f"\n=== Ablation runs: {len(tasks)} ===")
    run_grid(tasks)
else:
    print("RUN_ABLATION = False; skipping ablation runs.")

if RUN_DEPTH_SWEEP:
    tasks = [
        dict(experiment="depth", dataset=dataset, seed=seed, depth=depth, activation="entmax")
        for dataset in DATASETS
        for depth in DEPTHS
        for seed in TRIAL_SEEDS
    ]
    print(f"\n=== Depth-sweep runs: {len(tasks)} ===")
    run_grid(tasks)
else:
    print("RUN_DEPTH_SWEEP = False; skipping depth sweep.")


In [ ]:
df = read_results()

if df.empty:
    print("No results yet. Run the experiment cell first.")
else:
    pd.set_option("display.precision", 4)

    main_df = df[df["experiment"] == "main"]
    if not main_df.empty:
        main_summary = summarize(main_df, ["dataset"])
        main_summary["dataset"] = pd.Categorical(
            main_summary["dataset"], categories=DATASETS, ordered=True
        )
        main_summary = main_summary.sort_values("dataset")
        main_summary.to_csv(RESULTS_DIR / "summary_main.csv", index=False)
        print("\nMain GradTree results (mean ± std over trials)")
        display(main_summary)

    ablation_df = df[df["experiment"] == "ablation"]
    if not ablation_df.empty:
        ablation_summary = summarize(ablation_df, ["dataset", "activation"])
        ablation_summary.to_csv(RESULTS_DIR / "summary_ablation.csv", index=False)
        print("\nFeature-selection ablation")
        display(ablation_summary)

    depth_df = df[df["experiment"] == "depth"]
    if not depth_df.empty:
        depth_summary = summarize(depth_df, ["dataset", "depth"])
        depth_summary.to_csv(RESULTS_DIR / "summary_depth.csv", index=False)
        print("\nDepth sweep")
        display(depth_summary)

    print("Result CSVs:", sorted(p.name for p in RESULTS_DIR.glob("summary_*.csv")))


In [ ]:
df = read_results()
ablation_df = df[df["experiment"] == "ablation"] if not df.empty else df

if ablation_df.empty:
    print("No ablation results yet. Set RUN_ABLATION = True and run the experiment cell.")
else:
    ablation_summary = summarize(ablation_df, ["dataset", "activation"])
    y_positions = {
        activation: len(ABLATION_ACTIVATIONS) - 1 - i
        for i, activation in enumerate(ABLATION_ACTIVATIONS)
    }
    fig, axes = plt.subplots(1, len(ABLATION_DATASETS), figsize=(15, 5), sharey=True)
    fig.patch.set_facecolor(SURFACE)
    if len(ABLATION_DATASETS) == 1:
        axes = [axes]

    for ax, dataset in zip(axes, ABLATION_DATASETS):
        for activation in ABLATION_ACTIVATIONS:
            row = ablation_summary[
                (ablation_summary["dataset"] == dataset)
                & (ablation_summary["activation"] == activation)
            ]
            if row.empty:
                continue
            mean = float(row["test_macro_f1_mean"].iloc[0])
            std = float(row["test_macro_f1_std"].iloc[0])
            color = ACT_COLORS[activation]
            ax.errorbar(
                mean, y_positions[activation], xerr=std, fmt="o",
                color=color, ecolor=color, elinewidth=2, capsize=0,
                markersize=7, markeredgecolor=SURFACE, markeredgewidth=2,
                zorder=3,
            )
        if dataset in PAPER_TEST_F1:
            paper_mean, _ = PAPER_TEST_F1[dataset]
            ax.scatter(
                paper_mean, y_positions["entmax"], marker="D", s=70,
                facecolors=SURFACE, edgecolors=PAPER, linewidths=1.5, zorder=5,
            )
        style_axes(ax, grid_axis="x")
        ax.set_title(DATASET_LABELS[dataset], color=INK, fontsize=12, loc="left")
        ax.set_xlabel("Test macro F1", color=INK_2)
        ax.set_yticks(list(y_positions.values()))
        ax.set_yticklabels([ABLATION_LABELS[a] for a in ABLATION_ACTIVATIONS])
        ax.set_ylim(-0.6, len(ABLATION_ACTIVATIONS) - 0.4)

    handles = [
        Line2D([0], [0], color=ACT_COLORS[a], marker="o", lw=2, markersize=7, label=ABLATION_LABELS[a])
        for a in ABLATION_ACTIVATIONS
    ]
    handles.append(
        Line2D([0], [0], color=PAPER, marker="D", markerfacecolor=SURFACE, lw=0, markersize=7, label="Paper (Table 5)")
    )
    fig.legend(handles=handles, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(
        "Feature selection ablation: GradTree with each feature-choice method",
        x=0.01, y=1.04, ha="left", color=INK, fontsize=14,
    )
    fig.text(
        0.01, 0.95,
        "Mean ± std over trials. Gumbel-softmax and temperature annealing are not implemented.",
        color=MUTED, fontsize=9, ha="left",
    )
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig_feature_selection_ablation.png", dpi=200, bbox_inches="tight", facecolor=SURFACE)
    plt.show()


In [ ]:
df = read_results()
main_df = df[df["experiment"] == "main"] if not df.empty else df

if main_df.empty:
    print("No main results yet. Set RUN_MAIN = True and run the experiment cell.")
else:
    summary = summarize(main_df, ["dataset"])
    summary["dataset"] = pd.Categorical(summary["dataset"], categories=DATASETS, ordered=True)
    summary = summary.sort_values("dataset")

    fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
    fig.patch.set_facecolor(SURFACE)
    y_map = {dataset: len(DATASETS) - 1 - i for i, dataset in enumerate(DATASETS)}
    panels = [
        ("test_macro_f1_mean", "test_macro_f1_std", "Test macro F1", False, "test"),
        ("tree_nodes_mean", "tree_nodes_std", "Nodes after pruning", True, "size"),
        ("overfitting_mean", "overfitting_std", "Train F1 - test F1", False, None),
    ]

    for ax, (mean_col, std_col, title, log_x, paper_key) in zip(axes, panels):
        for dataset in DATASETS:
            row = summary[summary["dataset"] == dataset]
            if row.empty:
                continue
            y = y_map[dataset]
            mean = float(row[mean_col].iloc[0])
            std = float(row[std_col].iloc[0])
            ax.errorbar(
                mean, y, xerr=std, fmt="o", color=BLUE, ecolor=BLUE,
                elinewidth=2, capsize=0, markersize=7,
                markeredgecolor=SURFACE, markeredgewidth=2, zorder=3,
            )
            if paper_key == "test" and dataset in PAPER_TEST_F1:
                paper_mean, _ = PAPER_TEST_F1[dataset]
                ax.scatter(
                    paper_mean, y, marker="D", s=70, facecolors=SURFACE,
                    edgecolors=PAPER, linewidths=1.5, zorder=5,
                )
            if paper_key == "size" and dataset in PAPER_TREE_SIZE:
                paper_mean, _ = PAPER_TREE_SIZE[dataset]
                ax.scatter(
                    paper_mean, y, marker="D", s=70, facecolors=SURFACE,
                    edgecolors=PAPER, linewidths=1.5, zorder=5,
                )
        if log_x:
            ax.set_xscale("log")
            ax.set_xticks([10, 20, 50, 100, 200])
            ax.xaxis.set_major_formatter(ScalarFormatter())
            ax.xaxis.set_minor_formatter(NullFormatter())
        if title.startswith("Train F1"):
            ax.axvline(0.0, color=AXIS, lw=1, zorder=1)
        style_axes(ax, grid_axis="x")
        ax.set_title(title, color=INK, fontsize=12, loc="left")
        ax.set_yticks(list(y_map.values()))
        ax.set_yticklabels([DATASET_LABELS[d] for d in DATASETS])

    handles = [
        Line2D([0], [0], color=BLUE, marker="o", lw=2, markersize=7, label="GradTree (ours)"),
        Line2D([0], [0], color=PAPER, marker="D", markerfacecolor=SURFACE, lw=0, markersize=7, label="Paper (GradTree)"),
    ]
    fig.legend(handles=handles, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(
        "GradTree results with the paper's tuned settings (GradTree only)",
        x=0.01, y=1.04, ha="left", color=INK, fontsize=14,
    )
    fig.text(
        0.01, 0.95,
        "Mean ± std over trials. Paper diamonds: Tables 1/2 (F1) and Table 9 (tree size); "
        "paper overfitting is not reported per dataset.",
        color=MUTED, fontsize=9, ha="left",
    )
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig_main_results.png", dpi=200, bbox_inches="tight", facecolor=SURFACE)
    plt.show()


In [ ]:
df = read_results()
depth_df = df[df["experiment"] == "depth"] if not df.empty else df

if depth_df.empty:
    print("No depth-sweep results yet. Set RUN_DEPTH_SWEEP = True and run the experiment cell.")
else:
    depth_summary = summarize(depth_df, ["dataset", "depth"])
    fig, axes = plt.subplots(2, len(DATASETS), figsize=(20, 8), sharex=True)
    fig.patch.set_facecolor(SURFACE)

    for col, dataset in enumerate(DATASETS):
        ax_top = axes[0, col]
        ax_bottom = axes[1, col]
        subset = depth_summary[depth_summary["dataset"] == dataset].sort_values("depth")
        if subset.empty:
            ax_top.set_visible(False)
            ax_bottom.set_visible(False)
            continue

        depths = subset["depth"].astype(int).to_numpy()
        f1_mean = subset["test_macro_f1_mean"].to_numpy(dtype=float)
        f1_std = subset["test_macro_f1_std"].to_numpy(dtype=float)
        size_mean = subset["tree_nodes_mean"].to_numpy(dtype=float)
        size_std = subset["tree_nodes_std"].to_numpy(dtype=float)

        ax_top.plot(
            depths, f1_mean, color=BLUE, lw=2, marker="o", markersize=6,
            markeredgecolor=SURFACE, markeredgewidth=1.5, zorder=3,
        )
        ax_top.fill_between(
            depths, f1_mean - f1_std, f1_mean + f1_std,
            color=BLUE, alpha=0.16, linewidth=0, zorder=2,
        )

        ax_bottom.plot(
            depths, size_mean, color=BLUE, lw=2, marker="o", markersize=6,
            markeredgecolor=SURFACE, markeredgewidth=1.5, zorder=3,
        )
        ax_bottom.fill_between(
            depths, np.maximum(size_mean - size_std, 0.5), size_mean + size_std,
            color=BLUE, alpha=0.16, linewidth=0, zorder=2,
        )
        ax_bottom.set_yscale("log")

        tuned_depth = PAPER_SETTINGS[dataset]["depth"]
        line_kwargs = dict(color=MUTED, lw=1.5, zorder=1)
        if col == 0:
            line_kwargs["label"] = "GradTree tuned depth"
        ax_top.axvline(tuned_depth, **line_kwargs)
        ax_bottom.axvline(tuned_depth, **line_kwargs)

        style_axes(ax_top, grid_axis="y")
        style_axes(ax_bottom, grid_axis="y")
        ax_top.set_title(DATASET_LABELS[dataset], color=INK, fontsize=12, loc="left")
        ax_bottom.set_xticks(DEPTHS)
        ax_bottom.set_xlabel("Maximum depth", color=INK_2)
        if col == 0:
            ax_top.set_ylabel("Test macro F1", color=INK_2)
            ax_bottom.set_ylabel("Tree size (nodes, log scale)", color=INK_2)

    handles = [
        Line2D([0], [0], color=BLUE, marker="o", lw=2, markersize=6, label="GradTree (ours)"),
        Line2D([0], [0], color=MUTED, lw=1.5, label="GradTree tuned depth"),
    ]
    fig.legend(handles=handles, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(
        "Effect of maximum depth (GradTree only; mean ± std over trials)",
        x=0.01, y=1.04, ha="left", color=INK, fontsize=14,
    )
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig_depth_sweep.png", dpi=200, bbox_inches="tight", facecolor=SURFACE)
    plt.show()


# Notes and caveats

- **Scope:** GradTree only. CART and Random Forest are intentionally omitted, as requested.
- **Quick vs paper protocol:** `QUICK_RUN = True` uses 2 seeds and short training to validate the notebook. Set `QUICK_RUN = False` for the paper protocol: 10 seeds, 10,000 epochs, patience 200, 3 restarts.
- **Caching:** every run is appended to `model/sunny/results/runs.csv`, scoped to `PROTOCOL` (quick/paper, epochs, patience, restarts, and a hash of `PAPER_SETTINGS`). Changing those settings creates a new cache scope. Delete `runs.csv` to force a full rerun.
- **Feature-selection ablation:** this implementation supports `entmax`, `softmax`, and `sparsemax`. The paper's Gumbel-softmax and temperature-annealing variants are not implemented, so they are not shown.
- **Paper values:** test macro F1 from Tables 1/2, tree size from Table 9, tuned depth/settings from Table 12, and ST-entmax ablation values from Table 5. Paper overfitting is not reported per dataset, so no paper diamonds are drawn in the overfitting panel.
- **Test metric:** each trial uses the test split belonging to its data seed (42 + trial), with model seed 42. Per-dataset mean/std is over trials; the paper does not average macro F1 across datasets.
- **Data requirement:** all five datasets need `preprocess/<dataset>_output/seed_42` through `seed_51`. The dataset-check cell reports missing folders.
- **Execution:** this notebook was generated but not executed end-to-end here. Run the setup and dataset-check cells first, then use quick mode for a smoke pass before the full paper protocol.
